In [1]:
import os,pickle,numpy as np,pandas as pd,matplotlib.pyplot as plt,networkx as nx
from pathlib import Path
from scipy.stats import rankdata
from sklearn.covariance import LedoitWolf
from sklearn.metrics import adjusted_mutual_info_score
from networkx.algorithms.community import louvain_communities,modularity
import warnings
warnings.filterwarnings("ignore")

OUT=Path("rank_2");OUT.mkdir(exist_ok=True)
ST=["Stage I","Stage II","Stage III"];ORDER=["I-II","I-III","II-III"]
PAIRS=[("I-II",ST[0],ST[1]),("I-III",ST[0],ST[2]),("II-III",ST[1],ST[2])]
METHODS=["Pearson","Spearman","Partial-LW","MutualInfo","NEXUS"]
DENS=[5,7.5,10,12.5,15];NSEED=50;MI_MAX_T=100000;MI_Q=5;SEED=42;MAX_T=None

PKL={
ST[0]:"./result/wo_layer/stage1/binned_traces_stage_1.pkl",
ST[1]:"./result/wo_layer/stage2/binned_traces_stage_2.pkl",
ST[2]:"./result/wo_layer/stage3/binned_traces_stage_3.pkl"}

NP={
ST[0]:("./result/wo_layer/stage1/tuning/bs16_dp0.1_sw1.0/dynamic_full_graphs.npy","./result/wo_layer/stage1/df_scaled.csv"),
ST[1]:("./result/wo_layer/stage2/tuning/bs8_dp0.1_sw0.7/dynamic_full_graphs.npy","./result/wo_layer/stage1/df_scaled.csv"),
ST[2]:("./result/wo_layer/stage3/tuning/bs32_dp0.1_sw0.7/dynamic_full_graphs.npy","./result/wo_layer/stage1/df_scaled.csv")}


# ================= LOAD DATA =================
def loadN(g,l):
    A=np.load(g).mean(0);L=pd.read_csv(l).iloc[:,0].astype(str).str.strip().values
    np.fill_diagonal(A,0);return A,L

raw={s:loadN(*NP[s]) for s in ST}
common=[r for r in raw[ST[0]][1] if all(r in set(raw[s][1]) for s in ST)]

def reorder(A,L):
    d={r:i for i,r in enumerate(L)};i=[d[r] for r in common]
    return A[np.ix_(i,i)]

FC={"NEXUS":{s:reorder(*raw[s]) for s in ST}}

def getX(p):
    with open(p,"rb") as f:d=pickle.load(f)
    b=d["binned_traces"]
    miss=[r for r in common if r not in b]
    if miss:raise ValueError(f"Missing regions: {miss}")
    X=np.nan_to_num(np.array([np.asarray(b[r],float) for r in common]),nan=0)
    if MAX_T and X.shape[1]>MAX_T:X=X[:,np.linspace(0,X.shape[1]-1,MAX_T).astype(int)]
    if np.any(np.isclose(X.std(1),0)):raise ValueError(f"Zero-variance region: {p}")
    return X

XR={s:getX(PKL[s]) for s in ST}
for s in ST:print(s,XR[s].shape)
for m in METHODS[:-1]:FC[m]={}

# ================= TRADITIONAL FC =================
def prec2pc(Q):
    d=np.sqrt(np.clip(np.diag(Q),1e-15,None));A=-Q/np.outer(d,d)
    np.fill_diagonal(A,0);return np.clip(A,-1,1)

for s in ST:
    X=XR[s];Z=((X-X.mean(1,keepdims=True))/(X.std(1,keepdims=True)+1e-12)).T

    A=np.corrcoef(X);np.fill_diagonal(A,0);FC["Pearson"][s]=A
    A=np.corrcoef(rankdata(X,axis=1));np.fill_diagonal(A,0);FC["Spearman"][s]=A
    FC["Partial-LW"][s]=prec2pc(LedoitWolf().fit(Z).precision_)

# ================= ZERO-AWARE MI =================
rng=np.random.default_rng(SEED)
MI_N=min(MI_MAX_T,min(XR[s].shape[1] for s in ST))
XM={s:XR[s][:,np.sort(rng.choice(XR[s].shape[1],MI_N,replace=False))] for s in ST}

EDGES=[]
for r in range(len(common)):
    z=np.concatenate([XM[s][r][XM[s][r]!=0] for s in ST])
    EDGES.append(np.unique(np.quantile(z,np.arange(1,MI_Q)/MI_Q)) if len(z)>1 else np.array([]))

def discretize(X):
    B=np.zeros(X.shape,dtype=np.int16)
    for r,e in enumerate(EDGES):
        m=X[r]!=0;B[r,m]=np.digitize(X[r,m],e)+1
    return B

def nmi(x,y):
    k=max(int(x.max()),int(y.max()))+1
    c=np.bincount(x*k+y,minlength=k*k).reshape(k,k).astype(float);p=c/c.sum()
    px=p.sum(1);py=p.sum(0);d=px[:,None]*py[None,:];m=p>0
    mi=np.sum(p[m]*np.log(p[m]/d[m]))
    hx=-np.sum(px[px>0]*np.log(px[px>0]));hy=-np.sum(py[py>0]*np.log(py[py>0]))
    return 2*mi/(hx+hy) if hx+hy>0 else 0

def mi_matrix(B):
    n=len(B);A=np.zeros((n,n))
    for i in range(n):
        for j in range(i+1,n):A[i,j]=A[j,i]=nmi(B[i],B[j])
    return A

for s in ST:
    FC["MutualInfo"][s]=mi_matrix(discretize(XM[s]))
    x=FC["MutualInfo"][s][np.triu_indices(len(common),1)]
    print(f"{s}: MI n={MI_N}, mean={x.mean():.4f}, median={np.median(x):.4f}, P95={np.percentile(x,95):.4f}")

# ================= SC =================
sc=pd.read_excel("../SEU_net_wo_corticalLayers.xlsx",header=1,index_col=1).iloc[:,1:]
sc.index=sc.index.astype(str).str.strip();sc.columns=sc.columns.astype(str).str.strip()
SC=sc.loc[common,common].astype(float).values;np.fill_diagonal(SC,0)
SUP=(SC>0)|(SC.T>0);np.fill_diagonal(SUP,False)

# ================= METRICS =================
def vec(A,B,d):
    if d:
        m=~np.eye(len(A),dtype=bool);m&=((A!=0)|(B!=0));return A[m],B[m]
    i=np.triu_indices_from(A,1);return A[i],B[i]

def sim(A,B,d):
    a,b=vec(A,B,d);return np.corrcoef(a,b)[0,1]

def change(A,B,d,p=5):
    a,b=vec(A,B,d);x=np.abs(a-b);k=max(1,int(np.ceil(len(x)*p/100)))
    return np.sort(x)[-k:].sum()/x.sum()

def UW(A,d):
    W=(np.abs(A)+np.abs(A.T))/2 if d else np.abs(A.copy())
    np.fill_diagonal(W,0);return W

def topmask(W,p):
    i=np.triu_indices_from(W,1);x=W[i];k=max(1,int(np.ceil(len(x)*p/100)))
    q=np.argpartition(x,-k)[-k:];m=np.zeros(len(x),bool);m[q]=True;return m

def preserve(A,B,d,p):
    a=topmask(UW(A,d),p);b=topmask(UW(B,d),p)
    return np.sum(a&b)/np.sum(a)*100

def prop(W,p):
    i=np.triu_indices_from(W,1);x=W[i];k=max(1,int(np.ceil(len(x)*p/100)))
    q=np.argpartition(x,-k)[-k:];A=np.zeros_like(W);A[i[0][q],i[1][q]]=x[q]
    return A+A.T

def louv(W):
    G=nx.from_numpy_array(W);best=None;qb=-np.inf
    for seed in range(NSEED):
        c=louvain_communities(G,weight="weight",seed=seed)
        q=modularity(G,c,weight="weight")
        if q>qb:best,qb=c,q
    z=np.empty(len(W),int)
    for k,c in enumerate(best):z[list(c)]=k
    return z

# ================= RESULTS =================
rows=[]
for m in METHODS:
    d=m=="NEXUS"
    for c,a,b in PAIRS:
        rows.append([m,c,sim(FC[m][a],FC[m][b],d),change(FC[m][a],FC[m][b],d)*100])
res=pd.DataFrame(rows,columns=["method","comparison","similarity","top5_change"])

rr=[]
for m in METHODS:
    d=m=="NEXUS"
    for c,a,b in PAIRS:
        for p in DENS:rr.append([m,c,p,preserve(FC[m][a],FC[m][b],d,p)])
edge=pd.DataFrame(rr,columns=["method","comparison","density","preservation"])

cc=[]
for den in DENS:
    for m in METHODS:
        d=m=="NEXUS";lab={s:louv(prop(UW(FC[m][s],d),den)) for s in ST}
        for c,a,b in PAIRS:cc.append([m,c,den,adjusted_mutual_info_score(lab[a],lab[b])])
comm=pd.DataFrame(cc,columns=["method","comparison","density","AMI"])

idx=np.triu_indices_from(SUP,1);bg=SUP[idx].mean();ss=[]
for m in METHODS:
    for s in ST:
        if m=="NEXUS":v=1.
        else:
            W=UW(FC[m][s],False);x=W[idx];k=max(1,int(np.ceil(len(x)*.05)))
            q=np.argpartition(x,-k)[-k:];v=SUP[idx][q].mean()
        ss.append([m,s,v*100])
sup=pd.DataFrame(ss,columns=["method","stage","SC_support"])

for d,n in [(res,"benchmark_summary"),(edge,"benchmark_edge_preservation"),
            (comm,"benchmark_community"),(sup,"benchmark_sc_support")]:
    d.to_csv(OUT/f"{n}.csv",index=False)

print("\nSimilarity\n",res.pivot(index="method",columns="comparison",values="similarity").loc[METHODS,ORDER].round(3))
print("\nTop5 change (%)\n",res.pivot(index="method",columns="comparison",values="top5_change").loc[METHODS,ORDER].round(1))
print("\nEdge preservation (%)\n",edge.groupby(["method","density"]).preservation.mean().unstack().loc[METHODS].round(1))
print("\nMean AMI\n",comm.groupby(["method","density"]).AMI.mean().unstack().loc[METHODS].round(3))
print("\nSC support (%)\n",sup.pivot(index="method",columns="stage",values="SC_support").loc[METHODS,ST].round(1))

# ================= FIGURE =================
PLOT_METHODS=["NEXUS","Pearson","Spearman","Partial-LW","MutualInfo"]
plt.rcParams.update({"font.size":8,"axes.labelsize":9,
                     "xtick.labelsize":8,"ytick.labelsize":8,
                     "axes.linewidth":.8,"pdf.fonttype":42,"ps.fonttype":42})

COL={"Pearson":"#64748B","Spearman":"#819CB4","Partial-LW":"#568E7C",
     "MutualInfo":"#88738F","NEXUS":"#C25F3D"}

fig=plt.figure(figsize=(12.6,7))
gs=fig.add_gridspec(2,3,left=.06,right=.985,top=.96,bottom=.10,wspace=.28,hspace=.34)
A,B,C,D,E,F=[fig.add_subplot(gs[i,j]) for i,j in [(0,0),(0,1),(0,2),(1,0),(1,1),(1,2)]]

def panel(ax,s):ax.text(-.10,1.04,s,transform=ax.transAxes,fontsize=13,fontweight="bold")
def clean(ax):ax.spines[["top","right"]].set_visible(False);ax.tick_params(length=3)

def heat(ax,df,vmin,vmax,cmap,fmt,suf=""):
    im=ax.imshow(df.values,aspect="auto",vmin=vmin,vmax=vmax,cmap=cmap)
    ax.set_yticks(range(len(df)),df.index);ax.set_xticks(range(df.shape[1]),df.columns)
    for i in range(len(df)):
        for j in range(df.shape[1]):
            v=df.iloc[i,j]
            ax.text(j,i,f"{v:{fmt}}{suf}",ha="center",va="center",fontsize=7,
                    color="white" if (v-vmin)/(vmax-vmin)>.62 else "#222")
    for x in ax.spines.values():x.set_visible(False)
    return im

panel(A,"A");A.axis("off");A.set(xlim=(0,10),ylim=(0,10))
desc={"Pearson":"Linear marginal dependence","Spearman":"Rank-based dependence",
      "Partial-LW":"Regularized conditional dependence",
      "MutualInfo":"Nonlinear dependence (normalized MI)",
      "NEXUS":"Anatomy-constrained dynamic graph"}

for i,m in enumerate(METHODS):
    y=8.7-i*1.65
    A.scatter(.8,y,s=75,color=COL[m])
    A.text(1.35,y+.18,m,fontsize=9,fontweight="bold",va="center")
    A.text(1.35,y-.35,desc[m],fontsize=7.5,color="#666",va="center")
A.text(.8,.35,"All methods use the same regional population signals.",fontsize=7.5,color="#555")

panel(B,"B")
x=res.pivot(index="method",columns="comparison",values="similarity").loc[PLOT_METHODS,ORDER]
x.columns=["I–II","I–III","II–III"]
im=heat(B,x,-1,1,"RdBu_r",".2f")
B.set_title("Cross-stage FC similarity",fontsize=9,pad=8)
fig.colorbar(im,ax=B,fraction=.035,pad=.025,label="Matrix correlation")

panel(C,"C")
for m in METHODS:
    z=edge[edge.method==m].groupby("density").preservation.agg(["mean","min","max"]).reindex(DENS)
    C.fill_between(DENS,z["min"],z["max"],color=COL[m],alpha=.07)
    C.plot(DENS,z["mean"],"-o",color=COL[m],lw=2.3 if m=="NEXUS" else 1.35,ms=3.5,label=m)
C.set(xlabel="Retained strongest edges (%)",ylabel="Strong-edge preservation (%)",ylim=(0,100))
C.set_xticks(DENS);clean(C);C.legend(frameon=False,fontsize=6.8,ncol=2,loc="best")

panel(D,"D")
for m in METHODS:
    z=comm[comm.method==m].groupby("density").AMI.agg(["mean","min","max"]).reindex(DENS)
    D.fill_between(DENS,z["min"],z["max"],color=COL[m],alpha=.07)
    D.plot(DENS,z["mean"],"-o",color=COL[m],
           lw=2.3 if m=="NEXUS" else 1.35,ms=3.5,label=m)

D.set(xlabel="Retained strongest edges (%)",
      ylabel="Community preservation (AMI)",ylim=(0,1.02))
D.set_xticks(DENS);clean(D)
D.legend(frameon=False,fontsize=6.8,ncol=2,loc="best")

panel(E,"E")
x=res.pivot(index="method",columns="comparison",values="top5_change").loc[PLOT_METHODS,ORDER]
x.columns=["I–II","I–III","II–III"]
im=heat(E,x,0,100,"magma",".1f","%")
E.set_title("Change captured by top 5% edges",fontsize=9,pad=8)
fig.colorbar(im,ax=E,fraction=.035,pad=.025,label="% total change")

panel(F,"F")
x=sup.pivot(index="method",columns="stage",values="SC_support").loc[PLOT_METHODS,ST]
x.columns=["Stage I","Stage II","Stage III"]
im=heat(F,x,0,100,"cividis",".1f","%")
F.set_title("Direct anatomical support of top 5% FC",fontsize=9,pad=8)
fig.colorbar(im,ax=F,fraction=.035,pad=.025,label="SC-supported edges (%)")
F.text(.99,-.15,f"Background SC support: {bg*100:.1f}%; NEXUS = 100% by construction",
       transform=F.transAxes,ha="right",fontsize=6.8,color="#555")

plt.savefig(OUT/"benchmark_main.png",dpi=600,bbox_inches="tight")
plt.savefig(OUT/"benchmark_main.pdf",bbox_inches="tight")
plt.close()

Stage I (89, 1000)
Stage II (89, 1000)
Stage III (89, 1000)
Stage I: MI n=1000, mean=0.0504, median=0.0260, P95=0.1633
Stage II: MI n=1000, mean=0.0240, median=0.0127, P95=0.0826
Stage III: MI n=1000, mean=0.1902, median=0.1568, P95=0.4798

Similarity
 comparison   I-II  I-III  II-III
method                          
Pearson     0.772  0.583   0.533
Spearman    0.537  0.378   0.359
Partial-LW  0.141  0.112   0.065
MutualInfo  0.577  0.592   0.496
NEXUS       0.603  0.555   0.473

Top5 change (%)
 comparison  I-II  I-III  II-III
method                         
Pearson     18.7   13.5    12.2
Spearman    19.1   10.8    11.3
Partial-LW  15.0   15.6    16.2
MutualInfo  23.2   14.8    14.8
NEXUS       40.0   37.2    35.6

Edge preservation (%)
 density     5.0   7.5   10.0  12.5  15.0
method                                  
Pearson     43.2  45.9  47.1  50.1  52.1
Spearman    20.7  28.6  34.5  38.2  41.3
Partial-LW  13.1  13.6  15.8  17.8  18.8
MutualInfo  38.6  41.4  45.3  48.2  49.5
NEXU

In [2]:
# ================= FIGURE =================
from matplotlib.patches import FancyBboxPatch,FancyArrowPatch

plt.rcParams.update({"font.size":8,"axes.labelsize":9,"xtick.labelsize":8,"ytick.labelsize":8,
                     "axes.linewidth":.8,"pdf.fonttype":42,"ps.fonttype":42})
COL={"Pearson":"#64748B","Spearman":"#819CB4","Partial-LW":"#568E7C","MutualInfo":"#88738F","NEXUS":"#C25F3D"}
CB,CN="#65758B","#C25F3D"

fig=plt.figure(figsize=(12.6,7))
gs=fig.add_gridspec(2,3,left=.06,right=.985,top=.96,bottom=.10,wspace=.35,hspace=.30)
A,B,C,D,E,F=[fig.add_subplot(gs[i,j]) for i,j in [(0,0),(0,1),(0,2),(1,0),(1,1),(1,2)]]

def panel(ax,s):ax.text(-.09,1.035,s,transform=ax.transAxes,fontsize=13,fontweight="bold")
def clean(ax):ax.spines[["top","right"]].set_visible(False);ax.tick_params(length=3)
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

def heat(ax,df,vmin,vmax,cmap,fmt,suf="",fs=7.8):
    cm = plt.get_cmap(cmap)
    norm = Normalize(vmin=vmin, vmax=vmax)

    nrow, ncol = df.shape

    for i in range(nrow):
        for j in range(ncol):
            v = df.iloc[i,j]
            c = cm(norm(v))

            ax.add_patch(
                plt.Rectangle(
                    (j-.5, i-.5), 1, 1,
                    facecolor=c,
                    edgecolor="none"
                )
            )

            r,g,b,_ = c
            lum = .2126*r + .7152*g + .0722*b

            ax.text(
                j,i,f"{v:{fmt}}{suf}",
                ha="center",va="center",
                fontsize=fs,fontweight="bold",
                color="black" if lum>.55 else "white"
            )

    ax.set_xlim(-.5,ncol-.5)
    ax.set_ylim(nrow-.5,-.5)

    ax.set_yticks(range(nrow),df.index)
    ax.set_xticks(range(ncol),df.columns)

    for q in ax.spines.values():
        q.set_visible(False)

    return ScalarMappable(norm=norm,cmap=cm)

# ===== chance-corrected edge preservation for panel C =====
EDGE_DENS=[2.5,5,7.5,10]
SC_DEN=SUP[np.triu_indices_from(SUP,1)].mean()*100

def edge_cc_fun(A,B,d,p):
    a,b=topmask(UW(A,d),p),topmask(UW(B,d),p)
    obs=(a&b).sum()/a.sum()*100
    chance=min(100,p/(SC_DEN/100) if d else p)
    return (obs-chance)/(100-chance)*100 if chance<100 else np.nan

edge_cc=pd.DataFrame(
    [[m,c,p,edge_cc_fun(FC[m][a],FC[m][b],m=="NEXUS",p)]
     for m in METHODS for c,a,b in PAIRS for p in EDGE_DENS],
    columns=["method","comparison","density","preservation"]
)

print("\nChance-corrected edge preservation (%)")
print(edge_cc.groupby(["method","density"]).preservation.mean().unstack().loc[METHODS].round(1))

# ================= A: BENCHMARK DESIGN =================
panel(A,"A");A.axis("off");A.set(xlim=(0,10),ylim=(0,10))

def box(ax,x,y,w,h,t,ec="#777",fc="#FAFAFA",fs=7.6,weight="normal"):
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle="round,pad=.035,rounding_size=.08",fc=fc,ec=ec,lw=.85))
    ax.text(x+w/2,y+h/2,t,ha="center",va="center",fontsize=fs,fontweight=weight,linespacing=1.18)

def arr(ax,x1,y1,x2,y2):
    ax.add_patch(FancyArrowPatch((x1,y1),(x2,y2),arrowstyle="-|>",mutation_scale=8,lw=.8,color="#777"))

# conventional lane
A.text(.2,9.25,"Conventional FC estimators",fontsize=9,fontweight="bold",color=CB)
box(A,.35,6.55,2.05,1.25,"Regional\npopulation signals")
arr(A,2.5,7.18,2.95,7.18)

A.add_patch(FancyBboxPatch((3.05,5.75),4.55,2.85,boxstyle="round,pad=.04,rounding_size=.08",
                           fc="#F7F8FA",ec=CB,lw=.9))
for y,m,d in [(8.00,"Pearson","linear dependence"),(7.38,"Spearman","rank dependence"),
              (6.76,"Partial-LW","conditional dependence"),(6.14,"Mutual information","nonlinear dependence")]:
    A.scatter(3.38,y,s=23,color=COL["MutualInfo" if m=="Mutual information" else m],zorder=3)
    A.text(3.62,y,m,fontsize=7.3,fontweight="bold",va="center")
    A.text(5.35,y,d,fontsize=6.8,color="#666",va="center")

arr(A,7.72,7.18,8.12,7.18)
box(A,8.22,6.55,1.45,1.25,"FC\nundirected",ec=CB,fc="#F3F6F9",fs=7.5)

# NEXUS lane
A.text(.2,4.35,"NEXUS",fontsize=9,fontweight="bold",color=CN)
box(A,.35,2.65,2.05,1.10,"Regional\npopulation signals")
box(A,.35,.95,2.05,1.10,"Structural\nconnectome")
arr(A,2.5,3.20,3.10,2.80);arr(A,2.5,1.50,3.10,2.25)

box(A,3.20,1.55,4.15,2.05,"NEXUS\nSC-constrained · dynamic · directed",
    ec=CN,fc="#FFF7F3",fs=7.5,weight="bold")
arr(A,7.47,2.58,8.12,2.58)
box(A,8.22,1.95,1.45,1.25,"FC\ndirected",ec=CN,fc="#FFF7F3",fs=7.5)

A.text(.35,.25,"Same population-level neural representation used across methods.",
       fontsize=6.8,color="#666")

# ================= B =================
panel(B,"B")
x=res.pivot(index="method",columns="comparison",values="similarity").loc[PLOT_METHODS,ORDER]
x.columns=["I–II","I–III","II–III"]
im=heat(B,x,0,1,"YlGnBu",".2f")
B.set_title("Cross-stage FC similarity",fontsize=9,pad=8)
fig.colorbar(im,ax=B,fraction=.035,pad=.025,label="Matrix correlation")

# ================= C =================
panel(C,"C")
for m in METHODS:
    z=edge_cc[edge_cc.method==m].groupby("density").preservation.agg(["mean","min","max"]).reindex(EDGE_DENS)
    C.fill_between(EDGE_DENS,z["min"],z["max"],color=COL[m],alpha=.07,lw=0)
    C.plot(EDGE_DENS,z["mean"],"-o",color=COL[m],
           lw=2.3 if m=="NEXUS" else 1.35,ms=3.5,label=m)

C.axhline(0,color="#AAAAAA",ls="--",lw=.7)
C.set_xlabel("Retained strongest edges (%)")
C.set_ylabel("Chance-corrected\nedge preservation (%)")
C.set_xticks(EDGE_DENS)
C.set_ylim(min(-5,edge_cc.preservation.min()-5),100)
clean(C)
C.legend(frameon=False,fontsize=6.8,ncol=2,loc="best")

# ================= D =================
panel(D,"D") 
for m in METHODS: 
    z=comm[comm.method==m].groupby("density").AMI.agg(["mean","min","max"]).reindex(DENS) 
    
    D.fill_between(
        DENS,z["min"],z["max"],
        color=COL[m],alpha=.07,lw=0
    )
    
    D.plot(
        DENS,z["mean"],"-o",
        color=COL[m],
        lw=2.3 if m=="NEXUS" else 1.35,
        ms=3.5,label=m
    )

D.set(
    xlabel="Retained strongest edges (%)",
    ylabel="Community preservation (AMI)",
    ylim=(0,1.02)
)
D.set_xticks(DENS)
clean(D)
D.legend(frameon=False,fontsize=6.8,ncol=2,loc="best")

# ================= E =================
panel(E,"E")
x=res.pivot(index="method",columns="comparison",values="top5_change").loc[PLOT_METHODS,ORDER];x.columns=["I–II","I–III","II–III"]
im=heat(E,x,0,100,"magma",".1f","%");E.set_title("Change captured by top 5% edges",fontsize=12,pad=8)
fig.colorbar(im,ax=E,fraction=.035,pad=.025,label="% total change")

# ================= F =================
panel(F,"F")
x=sup.pivot(index="method",columns="stage",values="SC_support").loc[PLOT_METHODS,ST];x.columns=["Stage I","Stage II","Stage III"]
im=heat(F,x,0,100,"cividis",".1f","%");F.set_title("Direct anatomical support of top 5% FC",fontsize=12,pad=8)
fig.colorbar(im,ax=F,fraction=.035,pad=.025,label="SC-supported edges (%)")
F.text(.99,-.14,f"Background SC support: {bg*100:.1f}%; NEXUS = 100% by construction",
       transform=F.transAxes,ha="right",fontsize=6.8,color="#555")

plt.savefig(OUT/"benchmark_main.png",dpi=600,bbox_inches="tight",facecolor="white")
plt.savefig(OUT/"benchmark_main.pdf",bbox_inches="tight",facecolor="white")
plt.close()


Chance-corrected edge preservation (%)
density     2.5   5.0   7.5   10.0
method                            
Pearson     35.1  40.2  41.5  41.2
Spearman    14.5  16.6  22.8  27.2
Partial-LW  11.0   8.5   6.6   6.5
MutualInfo  33.0  35.4  36.6  39.2
NEXUS       68.1  82.9  88.2  85.2
